# 多任务学习 Multitask Learning

<img src="https://raw.githubusercontent.com/LisonEvf/practicalAI-cn/master/images/logo.png" width=150>

多任务学习是一种机器学习范式，通过共享表示同时学习多个相关任务，从而提升模型的泛化能力和效率。

Multitask learning is a machine learning paradigm that improves generalization by learning multiple related tasks simultaneously through shared representations.

<img src="https://raw.githubusercontent.com/LisonEvf/practicalAI-cn/master/images/multitask.png" width=500>

# 概述 Overview

* **目标:**  通过共享特征表示，同时学习多个相关任务。
* **优点:** 
  * 共享知识，提升泛化能力
  * 减少过拟合风险
  * 提高计算效率
* **缺点:**
  * 任务可能冲突（任务干扰）
  * 训练复杂度增加
  * 需要平衡任务权重
* **其他:** 
  * 广泛应用于NLP、CV、语音识别
  * 是迁移学习的重要分支

# 设置 Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt

# 多任务学习架构 Multitask Learning Architecture

In [ ]:
# 共享编码器 + 多任务解码器
# Shared encoder + multiple task decoders

class SharedEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(SharedEncoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
    
    def forward(self, x):
        return self.encoder(x)

class TaskDecoder(nn.Module):
    def __init__(self, hidden_dim, output_dim):
        super(TaskDecoder, self).__init__()
        self.decoder = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, output_dim)
        )
    
    def forward(self, x):
        return self.decoder(x)

class MultitaskModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, task_outputs):
        # task_outputs: dict, e.g., {'task1': 2, 'task2': 3}
        super(MultitaskModel, self).__init__()
        self.shared_encoder = SharedEncoder(input_dim, hidden_dim)
        self.task_decoders = nn.ModuleDict()
        for task_name, output_dim in task_outputs.items():
            self.task_decoders[task_name] = TaskDecoder(hidden_dim, output_dim)
    
    def forward(self, x, task_names):
        shared_repr = self.shared_encoder(x)
        outputs = {}
        for task_name in task_names:
            outputs[task_name] = self.task_decoders[task_name](shared_repr)
        return outputs

# 生成多任务数据 Generate Multitask Data

In [ ]:
# 生成示例数据：两个相关但不同的分类任务
# Generate synthetic data: two related but different classification tasks

np.random.seed(42)
n_samples = 1000
input_dim = 20

# 输入特征（共享）
X = np.random.randn(n_samples, input_dim)

# 任务1：二分类（基于前10个特征）
task1_labels = (X[:, :10].sum(axis=1) > 0).astype(int)

# 任务2：三分类（基于后10个特征）
task2_logits = X[:, 10:].sum(axis=1)
task2_labels = np.digitize(task2_logits, bins=[-2, 2])

# 转换为张量
X_tensor = torch.FloatTensor(X)
task1_tensor = torch.LongTensor(task1_labels)
task2_tensor = torch.LongTensor(task2_labels)

dataset = TensorDataset(X_tensor, task1_tensor, task2_tensor)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

print(f"Task 1 - Classes: {task1_labels.sum()} positive, {n_samples - task1_labels.sum()} negative")
print(f"Task 2 - Class distribution: {np.bincount(task2_labels)}")

# 训练多任务模型 Train Multitask Model

In [ ]:
# 初始化模型
model = MultitaskModel(
    input_dim=20,
    hidden_dim=64,
    task_outputs={'task1': 2, 'task2': 3}
)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# 训练循环
num_epochs = 20
task_losses = {'task1': [], 'task2': []}

for epoch in range(num_epochs):
    total_loss = 0
    for batch_X, batch_task1, batch_task2 in train_loader:
        optimizer.zero_grad()
        
        # 前向传播
        outputs = model(batch_X, ['task1', 'task2'])
        
        # 计算多任务损失（简单加权求和）
        loss1 = criterion(outputs['task1'], batch_task1)
        loss2 = criterion(outputs['task2'], batch_task2)
        loss = loss1 + loss2
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        task_losses['task1'].append(loss1.item())
        task_losses['task2'].append(loss2.item())
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss:.4f}")

# 可视化损失曲线 Visualize Loss Curves

In [ ]:
plt.figure(figsize=(10, 5))
for task_name, losses in task_losses.items():
    # 滑动平均
    window = 20
    smoothed = np.convolve(losses, np.ones(window)/window, mode='valid')
    plt.plot(smoothed, label=task_name)

plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('Multitask Learning Loss Curves')
plt.legend()
plt.grid(True)
plt.show()

# 评估多任务模型 Evaluate Multitask Model

In [ ]:
# 在测试集上评估
model.eval()
with torch.no_grad():
    # 生成测试数据
    n_test = 200
    X_test = torch.randn(n_test, input_dim)
    true_task1 = (X_test[:, :10].sum(axis=1) > 0).long()
    true_task2 = torch.LongTensor(np.digitize(X_test[:, 10:].sum(axis=1), bins=[-2, 2]))
    
    # 预测
    outputs = model(X_test, ['task1', 'task2'])
    pred_task1 = outputs['task1'].argmax(dim=1)
    pred_task2 = outputs['task2'].argmax(dim=1)
    
    # 计算准确率
    acc_task1 = (pred_task1 == true_task1).float().mean()
    acc_task2 = (pred_task2 == true_task2).float().mean()
    
    print(f"Task 1 Accuracy: {acc_task1:.4f}")
    print(f"Task 2 Accuracy: {acc_task2:.4f}")

# TODO

- 任务权重平衡 Task Weighting Strategies
- 硬共享 vs 软共享 Hard vs Soft Parameter Sharing
- 辅助任务学习 Auxiliary Task Learning
- 多任务学习的梯度平衡 Gradient Balancing for MTL